In [2]:
# --- Imports ---
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import ParameterGrid

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt
import sys
from pathlib import Path

# --- Paths ---
project_root = Path("..")
data_dir = project_root / "data"
processed_dir = data_dir / "processed"
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
print("Added to PYTHONPATH:", src_path)

# --- Load processed CUP data ---
X_train = np.load(processed_dir / "X_train_scaled.npy")
y_train = np.load(processed_dir / "y_train.npy")
X_val = np.load(processed_dir / "X_val_scaled.npy")
y_val = np.load(processed_dir / "y_val.npy")
X_test_int = np.load(processed_dir / "X_test_int_scaled.npy")
y_test_int = np.load(processed_dir / "y_test_int.npy")

X_test_blind = np.load(processed_dir / "X_test_blind_scaled.npy")
ids_ts = np.load(processed_dir / "ids_ts.npy")

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Internal test:", X_test_int.shape, y_test_int.shape)
print("Blind test:", X_test_blind.shape)


Added to PYTHONPATH: ..\src
Train: (300, 12) (300, 4)
Val: (100, 12) (100, 4)
Internal test: (100, 12) (100, 4)
Blind test: (1000, 12)


In [2]:
def mee(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Mean Euclidean Error for 4D targets.
    """
    diffs = y_pred - y_true  # shape (n_samples, 4)
    dist = np.linalg.norm(diffs, axis=1)
    return float(np.mean(dist))


def evaluate_model(name, model, X_train, y_train, X_val, y_val):
    """
    Fit model on train, compute MEE on train and val.
    Returns a results dict and predictions.
    """
    model.fit(X_train, y_train)
    y_tr_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)

    res = {
        "model": name,
        "mee_train": mee(y_train, y_tr_pred),
        "mee_val": mee(y_val, y_val_pred),
    }
    return res, y_tr_pred, y_val_pred


In [3]:
results = []

# --- Linear Regression baseline ---
lin_reg = LinearRegression()
res_lin, _, _ = evaluate_model("LinearRegression", lin_reg, X_train, y_train, X_val, y_val)
results.append(res_lin)
print("LinearRegression:", res_lin)

# --- Ridge Regression (L2 regularization, small alpha sweep) ---
from sklearn.linear_model import Ridge

ridge_alphas = [0.0, 0.01, 0.1, 1.0, 10.0]
for alpha in ridge_alphas:
    ridge = Ridge(alpha=alpha)
    name = f"Ridge(alpha={alpha})"
    res_ridge, _, _ = evaluate_model(name, ridge, X_train, y_train, X_val, y_val)
    results.append(res_ridge)
    print(name, ":", res_ridge)

baseline_df = pd.DataFrame(results)
baseline_df.sort_values("mee_val", inplace=True)
baseline_df


LinearRegression: {'model': 'LinearRegression', 'mee_train': 25.090721834055426, 'mee_val': 25.89321263404367}
Ridge(alpha=0.0) : {'model': 'Ridge(alpha=0.0)', 'mee_train': 25.090721834055426, 'mee_val': 25.89321263404366}
Ridge(alpha=0.01) : {'model': 'Ridge(alpha=0.01)', 'mee_train': 25.0902912806863, 'mee_val': 25.88985068352671}
Ridge(alpha=0.1) : {'model': 'Ridge(alpha=0.1)', 'mee_train': 25.086721657221048, 'mee_val': 25.860529055826984}
Ridge(alpha=1.0) : {'model': 'Ridge(alpha=1.0)', 'mee_train': 25.07230615082163, 'mee_val': 25.637935507122844}
Ridge(alpha=10.0) : {'model': 'Ridge(alpha=10.0)', 'mee_train': 25.213147949692797, 'mee_val': 25.108623442151337}


,model,mee_train,mee_val
5,Ridge(alpha=10.0),25.213148,25.108623
4,Ridge(alpha=1.0),25.072306,25.637936
3,Ridge(alpha=0.1),25.086722,25.860529
2,Ridge(alpha=0.01),25.090291,25.889851
1,Ridge(alpha=0.0),25.090722,25.893213
0,LinearRegression,25.090722,25.893213


In [7]:
import numpy as np

def mee(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Mean Euclidean Error for 4-dimensional CUP targets.
    """
    diffs = y_pred - y_true
    dist = np.linalg.norm(diffs, axis=1)
    return float(np.mean(dist))


In [8]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

gb_param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.7, 1.0],
}

gb_results = []

for n in gb_param_grid["n_estimators"]:
    for lr in gb_param_grid["learning_rate"]:
        for depth in gb_param_grid["max_depth"]:
            for subs in gb_param_grid["subsample"]:

                base_gb = GradientBoostingRegressor(
                    n_estimators=n,
                    learning_rate=lr,
                    max_depth=depth,
                    subsample=subs
                )

                gb = MultiOutputRegressor(base_gb)

                gb.fit(X_train, y_train)

                ytr = gb.predict(X_train)
                yv = gb.predict(X_val)

                gb_results.append({
                    "model": "GradientBoosting",
                    "n_estimators": n,
                    "learning_rate": lr,
                    "max_depth": depth,
                    "subsample": subs,
                    "mee_train": mee(y_train, ytr),
                    "mee_val": mee(y_val, yv)
                })

gb_df = pd.DataFrame(gb_results)
gb_df_sorted = gb_df.sort_values("mee_val").reset_index(drop=True)

gb_df_sorted.head(10)


,model,n_estimators,learning_rate,max_depth,subsample,mee_train,mee_val
0,GradientBoosting,200,0.05,4,0.7,5.627041,20.650727
1,GradientBoosting,200,0.05,4,1.0,5.152872,20.735356
2,GradientBoosting,300,0.05,4,1.0,2.909512,20.749452
3,GradientBoosting,300,0.05,4,0.7,3.452326,20.755999
4,GradientBoosting,300,0.10,4,1.0,0.577539,20.886153
5,GradientBoosting,200,0.10,4,1.0,1.603978,20.906207
6,GradientBoosting,300,0.05,3,0.7,6.632602,21.005487
7,GradientBoosting,100,0.10,4,1.0,5.050821,21.011060
8,GradientBoosting,100,0.05,4,0.7,10.034169,21.116334
9,GradientBoosting,100,0.05,4,1.0,9.237288,21.157081


In [9]:
# Best GB configuration
best_gb = gb_df_sorted.iloc[0]
best_gb


model            GradientBoosting
n_estimators                  200
learning_rate                0.05
max_depth                       4
subsample                     0.7
mee_train                5.627041
mee_val                 20.650727
Name: 0, dtype: object

In [12]:
svr_param_grid = {
    "C": [0.1, 1.0, 10.0, 100.0],
    "epsilon": [0.01, 0.05, 0.1],
    "gamma": ["scale", 0.01, 0.1],
    "kernel": ["rbf"],
}

svr_results = []

for params in ParameterGrid(svr_param_grid):
    base_svr = SVR(**params)
    svr = MultiOutputRegressor(base_svr)
    svr.fit(X_train, y_train)
    
    y_tr = svr.predict(X_train)
    y_v = svr.predict(X_val)
    
    res = {
        "model": "SVR",
        **params,
        "mee_train": mee(y_train, y_tr),
        "mee_val": mee(y_val, y_v),
    }
    svr_results.append(res)

svr_df = pd.DataFrame(svr_results)
svr_df_sorted = svr_df.sort_values("mee_val").reset_index(drop=True)
svr_df_sorted.head(10)


,model,C,epsilon,gamma,kernel,mee_train,mee_val
0,SVR,100.0,0.01,0.1,rbf,19.735113,22.303512
1,SVR,100.0,0.05,0.1,rbf,19.726805,22.304479
2,SVR,100.0,0.10,0.1,rbf,19.716932,22.304813
3,SVR,100.0,0.01,scale,rbf,20.081492,22.460142
4,SVR,100.0,0.05,scale,rbf,20.078741,22.466746
5,SVR,100.0,0.10,scale,rbf,20.075036,22.473120
6,SVR,10.0,0.01,0.1,rbf,22.462358,23.716577
7,SVR,10.0,0.05,0.1,rbf,22.461983,23.722449
8,SVR,10.0,0.10,0.1,rbf,22.461444,23.723950
9,SVR,10.0,0.01,scale,rbf,22.750802,23.817395


In [3]:
from mlp_advanced import train_mlp_advanced

mlp_search_space = {
    "hidden_layers": [
        [64, 32],
        [128, 64],
        [128, 64, 32],
        [256, 128, 64],
        [64, 64, 64]
    ],
    "lr": [0.001, 0.0005, 0.0003],
    "weight_decay": [0.0, 1e-4, 1e-3],
    "dropout": [0.0],  # dropout hurts on small data
    "batch_size": [32, 64],
    "seed": [0, 1],    # stability check
}

mlp_results = []

for hidden in mlp_search_space["hidden_layers"]:
    for lr in mlp_search_space["lr"]:
        for wd in mlp_search_space["weight_decay"]:
            for bs in mlp_search_space["batch_size"]:
                for seed in mlp_search_space["seed"]:
                    model, history, val_mee = train_mlp_advanced(
                        X_train, y_train,
                        X_val, y_val,
                        hidden_layers=hidden,
                        lr=lr,
                        weight_decay=wd,
                        batch_size=bs,
                        activation="relu",
                        dropout=0.0,
                        patience=300,
                        seed=seed,
                        verbose=False,
                    )

                    mlp_results.append({
                        "hidden_layers": hidden,
                        "lr": lr,
                        "weight_decay": wd,
                        "batch_size": bs,
                        "seed": seed,
                        "val_mee": val_mee,
                    })
                    print(f"Done {hidden}, lr={lr}, wd={wd}, bs={bs}, seed={seed} => {val_mee:.4f}")

mlp_df = pd.DataFrame(mlp_results)
mlp_df_sorted = mlp_df.sort_values("val_mee").reset_index(drop=True)
mlp_df_sorted.head(10)


Done [64, 32], lr=0.001, wd=0.0, bs=32, seed=0 => 21.3589
Done [64, 32], lr=0.001, wd=0.0, bs=32, seed=1 => 21.8116
Done [64, 32], lr=0.001, wd=0.0, bs=64, seed=0 => 21.3273
Done [64, 32], lr=0.001, wd=0.0, bs=64, seed=1 => 21.5919
Done [64, 32], lr=0.001, wd=0.0001, bs=32, seed=0 => 20.9660
Done [64, 32], lr=0.001, wd=0.0001, bs=32, seed=1 => 21.7702
Done [64, 32], lr=0.001, wd=0.0001, bs=64, seed=0 => 21.2674
Done [64, 32], lr=0.001, wd=0.0001, bs=64, seed=1 => 21.5415
Done [64, 32], lr=0.001, wd=0.001, bs=32, seed=0 => 20.7515
Done [64, 32], lr=0.001, wd=0.001, bs=32, seed=1 => 21.5101
Done [64, 32], lr=0.001, wd=0.001, bs=64, seed=0 => 21.0947
Done [64, 32], lr=0.001, wd=0.001, bs=64, seed=1 => 21.6544
Done [64, 32], lr=0.0005, wd=0.0, bs=32, seed=0 => 20.8386
Done [64, 32], lr=0.0005, wd=0.0, bs=32, seed=1 => 21.8843
Done [64, 32], lr=0.0005, wd=0.0, bs=64, seed=0 => 21.3322
Done [64, 32], lr=0.0005, wd=0.0, bs=64, seed=1 => 21.7766
Done [64, 32], lr=0.0005, wd=0.0001, bs=32, seed

,hidden_layers,lr,weight_decay,batch_size,seed,val_mee
0,"[128, 64, 32]",0.0005,0.0010,64,1,20.563248
1,"[256, 128, 64]",0.0010,0.0010,32,0,20.578503
2,"[256, 128, 64]",0.0010,0.0001,32,0,20.639946
3,"[64, 32]",0.0010,0.0010,32,0,20.751493
4,"[128, 64, 32]",0.0010,0.0001,32,1,20.802483
5,"[64, 32]",0.0005,0.0000,32,0,20.838585
6,"[128, 64]",0.0010,0.0000,32,0,20.854267
7,"[256, 128, 64]",0.0010,0.0010,64,0,20.865128
8,"[128, 64]",0.0010,0.0001,32,0,20.890303
9,"[128, 64, 32]",0.0003,0.0001,64,1,20.899538


In [4]:
best = mlp_df_sorted.iloc[0]
best


hidden_layers    [128, 64, 32]
lr                      0.0005
weight_decay             0.001
batch_size                  64
seed                         1
val_mee              20.563248
Name: 0, dtype: object

In [13]:
# --------------------------------------------
# FINAL MODEL COMPARISON ON INTERNAL TEST SET
# --------------------------------------------

import torch
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor

# Merge TRAIN + VAL
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.vstack([y_train, y_val])

comparison_rows = []

# --------------------------------------------------
# 1. SVR – Best model from SVR grid search
# --------------------------------------------------
if 'svr_df_sorted' in globals() and len(svr_df_sorted) > 0:
    best_svr = svr_df_sorted.iloc[0]

    base_svr = SVR(
        C=best_svr["C"],
        epsilon=best_svr["epsilon"],
        gamma=best_svr["gamma"],
        kernel=best_svr["kernel"],
    )

    svr_best = MultiOutputRegressor(base_svr)
    svr_best.fit(X_train_val, y_train_val)

    y_test_svr = svr_best.predict(X_test_int)

    comparison_rows.append({
        "family": "SVR",
        "config": dict(best_svr),
        "mee_test_int": mee(y_test_int, y_test_svr),
    })

# --------------------------------------------------
# 2. Gradient Boosting – Best GB model
# --------------------------------------------------
if 'gb_df_sorted' in globals() and len(gb_df_sorted) > 0:
    best_gb = gb_df_sorted.iloc[0]

    base_gb = GradientBoostingRegressor(
        n_estimators=int(best_gb["n_estimators"]),
        learning_rate=best_gb["learning_rate"],
        max_depth=int(best_gb["max_depth"]),
        subsample=best_gb["subsample"]
    )

    gb_best = MultiOutputRegressor(base_gb)
    gb_best.fit(X_train_val, y_train_val)

    y_test_gb = gb_best.predict(X_test_int)

    comparison_rows.append({
        "family": "GradientBoosting",
        "config": dict(best_gb),
        "mee_test_int": mee(y_test_int, y_test_gb)
    })

# --------------------------------------------------
# 3. Advanced MLP – Best neural model
# --------------------------------------------------
best_mlp = mlp_df_sorted.iloc[0]

mlp_model, mlp_hist, val_mee = train_mlp_advanced(
    X_train_val, y_train_val,
    X_test_int, y_test_int,
    hidden_layers=best_mlp["hidden_layers"],
    lr=best_mlp["lr"],
    weight_decay=best_mlp["weight_decay"],
    batch_size=int(best_mlp["batch_size"]),   # cast to python int
    activation="relu",
    dropout=0.0,
    seed=int(best_mlp["seed"]),               # cast to python int
    patience=300,
    verbose=False
)

# Predictions
X_test_tensor = torch.tensor(X_test_int, dtype=torch.float32)
with torch.no_grad():
    y_test_mlp = mlp_model(X_test_tensor).numpy()

comparison_rows.append({
    "family": "MLP_Advanced",
    "config": {
        "hidden_layers": best_mlp["hidden_layers"],
        "lr": best_mlp["lr"],
        "weight_decay": best_mlp["weight_decay"],
        "batch_size": int(best_mlp["batch_size"]),
        "seed": int(best_mlp["seed"]),
    },
    "mee_test_int": mee(y_test_int, y_test_mlp)
})

# --------------------------------------------------
# Final comparison table
# --------------------------------------------------
comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values("mee_test_int").reset_index(drop=True)

comparison_df


,family,config,mee_test_int
0,GradientBoosting,"{'model': 'GradientBoosting', 'n_estimators': ...",20.647344
1,MLP_Advanced,"{'hidden_layers': [128, 64, 32], 'lr': 0.0005,...",20.979399
2,SVR,"{'model': 'SVR', 'C': 100.0, 'epsilon': 0.01, ...",23.966914
